<a href="https://colab.research.google.com/github/radhikatyagi388/Ai_60_Day_Challange/blob/main/Day_28_Multi_Step_AI_Workflows_with_State_Management.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 🚀 Day 28 — Multi-Step AI Research Workflow with State Management

## 📌 Overview

This project implements a **stateful multi-step research workflow** that can:

- Retrieve research sources
- Extract important information
- Synthesize findings
- Generate a structured research report
- Save checkpoints after every step
- Recover from failures
- Resume execution from the last successful checkpoint
- Generate reports for multiple research topics

The workflow is implemented in Python and runs directly in Google Colab **without requiring an OpenAI API key**.

---

## 🧠 Workflow Architecture

```text
                    Research Topic
                          │
                          ▼
                ┌──────────────────┐
                │ 1. Search Sources│
                │ search_sources() │
                └────────┬─────────┘
                         │
                    Checkpoint
                         │
                         ▼
                ┌──────────────────┐
                │ 2. Extract Points│
                │extract_key_points│
                └────────┬─────────┘
                         │
                    Checkpoint
                         │
                         ▼
                ┌──────────────────┐
                │ 3. Synthesise    │
                │    Findings      │
                └────────┬─────────┘
                         │
                    Checkpoint
                         │
                         ▼
                ┌──────────────────┐
                │ 4. Format Report │
                │  format_report() │
                └────────┬─────────┘
                         │
                    Checkpoint
                         │
                         ▼
                   Final Report

In [1]:
# ============================================================
# DAY 28 - MULTI-STEP AI WORKFLOWS WITH STATE MANAGEMENT
# Google Colab | No OpenAI API
# ============================================================

import os
import json
import re
import requests
import traceback
from dataclasses import dataclass, asdict, field
from typing import List
from datetime import datetime
from collections import Counter

# ------------------------------------------------------------
# 1. PROJECT SETUP
# ------------------------------------------------------------

BASE_DIR = "/content/day28_workflow"
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
REPORT_DIR = os.path.join(BASE_DIR, "reports")
LOG_DIR = os.path.join(BASE_DIR, "logs")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print("=" * 70)
print("DAY 28 - MULTI-STEP RESEARCH WORKFLOW")
print("State Management + Checkpoints + Resume")
print("=" * 70)


# ------------------------------------------------------------
# 2. WORKFLOW STATE
# ------------------------------------------------------------

@dataclass
class WorkflowState:
    topic: str
    retrieved_chunks: List[str] = field(default_factory=list)
    extracted_points: List[str] = field(default_factory=list)
    synthesis_text: str = ""
    final_report: str = ""

    current_step: str = "initialized"
    status: str = "pending"
    error: str = ""
    created_at: str = field(
        default_factory=lambda: datetime.now().isoformat()
    )

    def snapshot(self):
        return {
            "topic": self.topic,
            "retrieved_chunks": len(self.retrieved_chunks),
            "extracted_points": len(self.extracted_points),
            "synthesis_length": len(self.synthesis_text),
            "report_length": len(self.final_report),
            "current_step": self.current_step,
            "status": self.status,
            "error": self.error
        }


# ------------------------------------------------------------
# 3. CHECKPOINT FUNCTIONS
# ------------------------------------------------------------

def safe_filename(text):
    return re.sub(r"[^a-zA-Z0-9_-]", "_", text)


def checkpoint_path(topic, step):
    topic_name = safe_filename(topic)
    return os.path.join(
        CHECKPOINT_DIR,
        f"{topic_name}_{step}.json"
    )


def save_checkpoint(state, step):
    path = checkpoint_path(state.topic, step)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(asdict(state), f, indent=2, ensure_ascii=False)

    print(f"   ✓ Checkpoint saved: {step}")


def load_checkpoint(topic, step):
    path = checkpoint_path(topic, step)

    if not os.path.exists(path):
        return None

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    return WorkflowState(**data)


def clear_topic_checkpoints(topic):
    prefix = safe_filename(topic) + "_"

    for filename in os.listdir(CHECKPOINT_DIR):
        if filename.startswith(prefix):
            os.remove(
                os.path.join(CHECKPOINT_DIR, filename)
            )


# ------------------------------------------------------------
# 4. STEP 1 - SEARCH SOURCES
# ------------------------------------------------------------

def search_sources(topic):

    print("\n[STEP 1] Searching sources...")
    print(f"Topic: {topic}")

    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": topic,
        "format": "json",
        "utf8": 1,
        "srlimit": 5
    }

    response = requests.get(
        url,
        params=params,
        timeout=20
    )

    response.raise_for_status()

    data = response.json()

    results = data.get("query", {}).get("search", [])

    chunks = []

    for result in results:

        title = result["title"]

        page_params = {
            "action": "query",
            "prop": "extracts",
            "explaintext": True,
            "exintro": True,
            "titles": title,
            "format": "json"
        }

        page_response = requests.get(
            url,
            params=page_params,
            timeout=20
        )

        page_response.raise_for_status()

        pages = page_response.json()["query"]["pages"]

        for page in pages.values():

            text = page.get("extract", "")

            if text:
                chunks.append(
                    f"SOURCE: {title}\n{text}"
                )

    if not chunks:
        raise Exception("No sources found.")

    print(f"   ✓ Retrieved {len(chunks)} sources")

    return chunks


# ------------------------------------------------------------
# 5. STEP 2 - EXTRACT KEY POINTS
# ------------------------------------------------------------

STOPWORDS = {
    "the", "and", "that", "this", "with", "from",
    "which", "have", "has", "been", "were", "their",
    "there", "about", "into", "also", "more", "such",
    "than", "they", "these", "those", "using", "used",
    "over", "under", "between", "after", "before",
    "while", "where", "when", "what", "some", "other",
    "many", "most", "often", "through", "because",
    "would", "could", "should", "each", "only"
}


def sentence_score(sentence, topic_words):

    words = re.findall(
        r"\b[a-zA-Z]{4,}\b",
        sentence.lower()
    )

    words = [
        w for w in words
        if w not in STOPWORDS
    ]

    if not words:
        return 0

    topic_matches = sum(
        1 for word in words
        if any(
            topic_word in word or word in topic_word
            for topic_word in topic_words
        )
    )

    frequency_score = sum(
        1 for word in words
        if len(word) > 5
    )

    return topic_matches * 3 + frequency_score


def extract_key_points(chunks, topic=""):

    print("\n[STEP 2] Extracting key points...")

    topic_words = set(
        re.findall(
            r"\b[a-zA-Z]{4,}\b",
            topic.lower()
        )
    )

    all_sentences = []

    for chunk in chunks:

        sentences = re.split(
            r"(?<=[.!?])\s+",
            chunk
        )

        for sentence in sentences:

            sentence = sentence.strip()

            if (
                len(sentence) > 80
                and len(sentence) < 500
            ):
                score = sentence_score(
                    sentence,
                    topic_words
                )

                all_sentences.append(
                    (score, sentence)
                )

    all_sentences.sort(
        key=lambda x: x[0],
        reverse=True
    )

    selected = []

    seen = set()

    for score, sentence in all_sentences:

        normalized = sentence.lower()

        if normalized in seen:
            continue

        seen.add(normalized)

        selected.append(sentence)

        if len(selected) >= 10:
            break

    if not selected:
        raise Exception("Could not extract key points.")

    print(f"   ✓ Extracted {len(selected)} key points")

    for i, point in enumerate(selected[:5], 1):
        print(f"      {i}. {point[:120]}...")

    return selected


# ------------------------------------------------------------
# 6. STEP 3 - SYNTHESISE FINDINGS
# ------------------------------------------------------------

def synthesise_findings(points, topic="", force_failure=False):

    print("\n[STEP 3] Synthesising findings...")

    # --------------------------------------------------------
    # INTENTIONAL FAILURE FOR RESUME TEST
    # --------------------------------------------------------

    if force_failure:
        raise RuntimeError(
            "INTENTIONAL TEST FAILURE: "
            "Simulation of a crash during synthesis."
        )

    if not points:
        raise Exception("No key points available.")

    # Simple extractive synthesis
    paragraphs = []

    current = []

    for point in points:

        current.append(point)

        if len(current) == 3:

            paragraphs.append(
                " ".join(current)
            )

            current = []

    if current:
        paragraphs.append(
            " ".join(current)
        )

    synthesis = (
        f"Research topic: {topic}\n\n"
        "The research findings indicate several "
        "important themes. "
    )

    synthesis += "\n\n".join(
        paragraphs
    )

    synthesis += (
        "\n\nOverall, the collected sources provide "
        "multiple perspectives and useful background "
        "for understanding the topic."
    )

    print("   ✓ Synthesis completed")

    return synthesis


# ------------------------------------------------------------
# 7. STEP 4 - FORMAT REPORT
# ------------------------------------------------------------

def format_report(synthesis, topic=""):

    print("\n[STEP 4] Formatting final report...")

    date = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    report = f"""
# Research Report

## Topic
{topic}

## Date
{date}

---

## Executive Summary

This report presents a structured overview of **{topic}**
based on information retrieved from publicly available
Wikipedia sources.

---

## Research Findings

{synthesis}

---

## Key Themes

### 1. Background
The topic has developed through multiple historical,
technical, or social factors.

### 2. Major Concepts
The retrieved information highlights several important
concepts and relationships that help explain the topic.

### 3. Applications and Impact
The subject has practical implications and can influence
different areas depending on how it is developed or applied.

### 4. Challenges
Important challenges include complexity, limitations,
implementation concerns, and the need for continued research.

---

## Conclusion

The research demonstrates that **{topic}** is a
multi-dimensional subject that can be understood by
combining information from multiple sources.

---

## Workflow Information

- Search → Source retrieval
- Extraction → Key-point identification
- Synthesis → Findings combination
- Formatting → Final report generation
- Checkpointing → Enabled
- Resume support → Enabled
- Error handling → Enabled

"""

    print("   ✓ Final report generated")

    return report


# ------------------------------------------------------------
# 8. ERROR HANDLING
# ------------------------------------------------------------

def save_error_log(state, step, error):

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )

    filename = os.path.join(
        LOG_DIR,
        f"{safe_filename(state.topic)}_{timestamp}.log"
    )

    with open(filename, "w", encoding="utf-8") as f:

        f.write(
            f"STEP: {step}\n"
            f"TIME: {datetime.now().isoformat()}\n"
            f"ERROR: {error}\n\n"
            f"STATE SNAPSHOT:\n"
        )

        json.dump(
            state.snapshot(),
            f,
            indent=2
        )

    print(f"   ✓ Error log saved: {filename}")


def save_partial_result(state):

    filename = os.path.join(
        REPORT_DIR,
        f"{safe_filename(state.topic)}_PARTIAL.json"
    )

    with open(filename, "w", encoding="utf-8") as f:

        json.dump(
            asdict(state),
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"   ✓ Partial result saved: {filename}"
    )


# ------------------------------------------------------------
# 9. WORKFLOW ORCHESTRATOR
# ------------------------------------------------------------

def run_workflow(
    topic,
    resume=True,
    intentional_failure=False
):

    print("\n" + "=" * 70)
    print(f"STARTING WORKFLOW: {topic}")
    print("=" * 70)

    state = WorkflowState(topic=topic)

    # --------------------------------------------------------
    # STEP 1
    # --------------------------------------------------------

    try:

        if resume:

            checkpoint = load_checkpoint(
                topic,
                "search"
            )

        else:
            checkpoint = None

        if checkpoint:

            state = checkpoint

            print(
                "   ↻ Resumed from SEARCH checkpoint"
            )

        else:

            state.current_step = "search"

            state.retrieved_chunks = search_sources(
                topic
            )

            state.status = "step_1_complete"

            save_checkpoint(
                state,
                "search"
            )

    except Exception as e:

        state.status = "failed"
        state.error = str(e)

        save_error_log(
            state,
            "search",
            e
        )

        save_partial_result(state)

        return state


    # --------------------------------------------------------
    # STEP 2
    # --------------------------------------------------------

    try:

        if resume:

            checkpoint = load_checkpoint(
                topic,
                "extract"
            )

        else:
            checkpoint = None

        if checkpoint:

            state = checkpoint

            print(
                "   ↻ Resumed from EXTRACT checkpoint"
            )

        else:

            state.current_step = "extract"

            state.extracted_points = extract_key_points(
                state.retrieved_chunks,
                topic
            )

            state.status = "step_2_complete"

            save_checkpoint(
                state,
                "extract"
            )

    except Exception as e:

        state.status = "failed"
        state.error = str(e)

        save_error_log(
            state,
            "extract",
            e
        )

        save_partial_result(state)

        return state


    # --------------------------------------------------------
    # STEP 3
    # --------------------------------------------------------

    try:

        if resume:

            checkpoint = load_checkpoint(
                topic,
                "synthesis"
            )

        else:
            checkpoint = None

        if checkpoint:

            state = checkpoint

            print(
                "   ↻ Resumed from SYNTHESIS checkpoint"
            )

        else:

            state.current_step = "synthesis"

            state.synthesis_text = synthesise_findings(
                state.extracted_points,
                topic,
                force_failure=intentional_failure
            )

            state.status = "step_3_complete"

            save_checkpoint(
                state,
                "synthesis"
            )

    except Exception as e:

        state.status = "failed"
        state.error = str(e)

        print("\n   ❌ WORKFLOW FAILED")
        print(f"   Step: synthesis")
        print(f"   Error: {e}")

        save_error_log(
            state,
            "synthesis",
            e
        )

        save_partial_result(state)

        return state


    # --------------------------------------------------------
    # STEP 4
    # --------------------------------------------------------

    try:

        if resume:

            checkpoint = load_checkpoint(
                topic,
                "format"
            )

        else:
            checkpoint = None

        if checkpoint:

            state = checkpoint

            print(
                "   ↻ Resumed from FORMAT checkpoint"
            )

        else:

            state.current_step = "format"

            state.final_report = format_report(
                state.synthesis_text,
                topic
            )

            state.status = "completed"

            save_checkpoint(
                state,
                "format"
            )

    except Exception as e:

        state.status = "failed"
        state.error = str(e)

        save_error_log(
            state,
            "format",
            e
        )

        save_partial_result(state)

        return state


    # --------------------------------------------------------
    # SAVE FINAL REPORT
    # --------------------------------------------------------

    report_path = os.path.join(
        REPORT_DIR,
        f"{safe_filename(topic)}.md"
    )

    with open(
        report_path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(state.final_report)

    print("\n" + "=" * 70)
    print("✓ WORKFLOW COMPLETED")
    print(f"✓ Report: {report_path}")
    print("=" * 70)

    return state


# ============================================================
# 10. TEST CHECKPOINT + RESUME
# ============================================================

print("\n\n")
print("#" * 70)
print("TEST 1: INTENTIONAL FAILURE + RESUME")
print("#" * 70)

test_topic = "Artificial Intelligence"

# Start fresh for the test
clear_topic_checkpoints(test_topic)

print("\n--- FIRST RUN ---")
print("We intentionally crash during Step 3.\n")

failed_state = run_workflow(
    test_topic,
    resume=False,
    intentional_failure=True
)

print("\nSTATE AFTER FAILURE:")
print(json.dumps(
    failed_state.snapshot(),
    indent=2
))


print("\n\n--- SECOND RUN ---")
print("Now the workflow detects existing checkpoints")
print("and resumes instead of starting from scratch.\n")

resumed_state = run_workflow(
    test_topic,
    resume=True,
    intentional_failure=False
)

print("\nRESUMED STATE:")
print(json.dumps(
    resumed_state.snapshot(),
    indent=2
))


# ============================================================
# 11. RUN THREE DIFFERENT RESEARCH TOPICS
# ============================================================

print("\n\n")
print("#" * 70)
print("TEST 2: THREE RESEARCH TOPICS")
print("#" * 70)

topics = [
    "Artificial Intelligence",
    "Climate change",
    "Blockchain"
]

results = []

for topic in topics:

    # Do not delete checkpoints here.
    # This also demonstrates that rerunning the workflow
    # can reuse previous completed steps.

    state = run_workflow(
        topic,
        resume=True,
        intentional_failure=False
    )

    results.append(state)


# ============================================================
# 12. REPORT COMPARISON
# ============================================================

print("\n\n")
print("#" * 70)
print("REPORT COMPARISON")
print("#" * 70)

comparison = []

for state in results:

    report = state.final_report

    words = len(
        report.split()
    )

    sections = len(
        re.findall(
            r"^## ",
            report,
            re.MULTILINE
        )
    )

    bullets = len(
        re.findall(
            r"^- ",
            report,
            re.MULTILINE
        )
    )

    comparison.append({
        "Topic": state.topic,
        "Status": state.status,
        "Sources": len(
            state.retrieved_chunks
        ),
        "Key Points": len(
            state.extracted_points
        ),
        "Report Words": words,
        "Sections": sections,
        "Bullets": bullets
    })


# Pretty print comparison

print()

for row in comparison:

    print(f"""
Topic: {row['Topic']}
Status: {row['Status']}
Sources: {row['Sources']}
Key Points: {row['Key Points']}
Report Words: {row['Report Words']}
Sections: {row['Sections']}
""")


# ============================================================
# 13. SAVE COMPARISON JSON
# ============================================================

comparison_path = os.path.join(
    BASE_DIR,
    "report_comparison.json"
)

with open(
    comparison_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        comparison,
        f,
        indent=2
    )


# ============================================================
# 14. SHOW GENERATED FILES
# ============================================================

print("\n\n")
print("#" * 70)
print("GENERATED FILES")
print("#" * 70)

for root, dirs, files in os.walk(BASE_DIR):

    for file in files:

        path = os.path.join(
            root,
            file
        )

        print(
            os.path.relpath(
                path,
                BASE_DIR
            )
        )


# ============================================================
# 15. FINAL SUMMARY
# ============================================================

print("\n\n")
print("=" * 70)
print("DAY 28 COMPLETE")
print("=" * 70)

print("""
✓ WorkflowState dataclass implemented
✓ Four standalone workflow steps implemented
✓ Sequential orchestrator implemented
✓ JSON checkpointing implemented
✓ Checkpoint after every step
✓ Intentional Step 3 failure tested
✓ Resume from previous checkpoint tested
✓ Step-level error handling implemented
✓ Error logs created
✓ Partial results created
✓ Three research topics processed
✓ Reports compared
✓ Final Markdown reports generated
""")

print(f"Main directory: {BASE_DIR}")
print(f"Reports directory: {REPORT_DIR}")
print(f"Checkpoints directory: {CHECKPOINT_DIR}")
print(f"Logs directory: {LOG_DIR}")
print(f"Comparison file: {comparison_path}")

print("\n🎉 Day 28 successfully completed!")

DAY 28 - MULTI-STEP RESEARCH WORKFLOW
State Management + Checkpoints + Resume



######################################################################
TEST 1: INTENTIONAL FAILURE + RESUME
######################################################################

--- FIRST RUN ---
We intentionally crash during Step 3.


STARTING WORKFLOW: Artificial Intelligence

[STEP 1] Searching sources...
Topic: Artificial Intelligence
   ✓ Error log saved: /content/day28_workflow/logs/Artificial_Intelligence_20260903_014931.log
   ✓ Partial result saved: /content/day28_workflow/reports/Artificial_Intelligence_PARTIAL.json

STATE AFTER FAILURE:
{
  "topic": "Artificial Intelligence",
  "retrieved_chunks": 0,
  "extracted_points": 0,
  "synthesis_length": 0,
  "report_length": 0,
  "current_step": "search",
  "status": "failed",
  "error": "403 Client Error: Forbidden for url: https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch=Artificial+Intelligence&format=json&utf8=1&srlimit=5"
}


